# Dupin quad test

Dupin cyclide patches on a conical quad mesh by rectangle propagation
(`LMesh/utils/dupin.py`, algorithm in `docs/PROPAGATION_REDESIGN.md`). Run the cells in order:

1. setup and viewer;
2. the optimisation's energy terms (parallelogram, right angle);
3. the mesh (`MESH_NAME`) and the seed rectangle, with sliders;
4. propagation → patches → cones → optimisation;
5. export to `notebooks/out/dupin/dupin_sample/`.

**Viewer:** `ws.init()` starts a kayviz viewer and prints its URL; open it in a browser. To share one viewer between several notebooks, run `python -m kayviz` in a terminal first.

In [ ]:
# ── Cell 1 — Initialization ──────────────────────────────────────────────────
# Paths, webscope viewer, imports, palette, and output directory.
import sys, os
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for sub in ("",):  # project root: LMesh/, Lconjugacy/ (hanan, kayviz are installed)
    p = os.path.join(PROJECT_ROOT, sub) if sub else PROJECT_ROOT
    if p not in sys.path:
        sys.path.insert(0, p)

import ipywidgets as widgets
from IPython.display import display

from kayviz.webscope import script as ws

from hanan.geometry.mesh import Mesh
from hanan.geometry.utils import (
    read_obj,
    normalize_vertices,
    unit,
    merge_meshes,
    merge_curves,
    circle_3d,
)
from LMesh.utils.dupin import (
    find_corner_faces,
    grid_coordinates,
    seed_direction_towards,
    seed_rectangle_at_barycentre,
    strip_min_edge_length,
    propagate_rectangles,
    build_rectangle_frames,
    get_rectangle,
    Dupin_patch,
    dupin_patch_boundary_curves,
)
from hanan.geometry.primitives import circle_3pts
from LMesh.utils.dupin import DupinPropagationError, ruled_strip
from hanan.geometry.io import write_obj

ws.init()
ws.clear()


PALETTE = {
    "faces":     (89/255, 93/255, 100/255),
    "rectangle": (102/255, 51/255, 0.0),
    "dupin":     (93/255, 6/255, 8/255),
    "corner":    (0.9, 0.7, 0.1),
    "seed":      (0.7, 0.0, 0.0),
    "strip":     (0.92, 0.92, 0.92),
    "circle":    (0.1, 0.6, 0.9),
}

# init = cool/desaturated, opt = warm/saturated, so the two passes are easy to
# tell apart when both are shown in the viewer at once.
PASS_COLORS = {
    "opt": {
        "rectangles":      (0.80, 0.0, 0.0),     # Red
        "cyclides":        (0.05, 0.0, 0.8),   # deep blue
        "cyclides_bezier": (0.05, 0.0, 0.8),   # teal
        "cones":           (0.5, 0.5, 0.5),   # gray
        "cones_bezier":    (0.5, 0.5, 0.5),   # muted teal-grey
        "circles":         (0.8, 0.45, 0.85),   # blue
    },
    "init": {
        "rectangles":      (0.85, 0.45, 0.10),   # orange
        "cyclides":        (0.75, 0.10, 0.12),   # deep red
        "cyclides_bezier": (0.95, 0.65, 0.15),   # amber
        "cones":           (0.95, 0.90, 0.80),   # warm off-white
        "cones_bezier":    (0.90, 0.75, 0.55),   # sand
        "circles":         (0.95, 0.25, 0.55),   # magenta
    },
}

DATA    = os.path.join(PROJECT_ROOT, "data")
OUT_DIR = os.path.join(PROJECT_ROOT, "notebooks", "out", "dupin", "dupin_sample")
os.makedirs(OUT_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output dir  :", OUT_DIR)
print("Open http://localhost:8000?app=script")

In [ ]:
# ── Cell 2 — Optimization energy terms ───────────────────────────────────────
# Two custom ObjectiveTerm subclasses used by the optimization in Cell 4.
# The other two terms (Cyclicity, Planarity-as-conicality) come from
# hanan.optimization directly.
#
#   Parallelogram  : each rectangle stays a parallelogram
#   StraightAngle  : right angle at each rectangle corner
#
# See docs/PROPAGATION_REDESIGN.md, "Optimization", for how these combine with
# Cyclicity (concircularity at each interior vertex) and conicality.
from hanan.optimization.objective_term import ObjectiveTerm
from hanan.optimization.indexing import interleave_indices_dim


class Parallelogram(ObjectiveTerm):

    def __init__(self) -> None:
        """ Rectangle: Energy term that minimize the deviation of a face from being a parallelogram.
        The energy function is given by:
        Energy :   sum_F (v2 + v0 - v3 - v1)
          where F is a  face with vertices v0, v1, v2, v3 
          v3 ------------ v2
          |                | 
          |                | 
          |                | 
          v0 ------------ v1
        """
        super().__init__()
        self.name = "Parallelogram" # Name of the constraint
        
        
      
    def initialize_objective(self, X, var_idx, var_name, faces ) -> None:
        """ 
        Input:
            X : Variables
            var_idx : dictionary of indices of variables
            var_name: Name of the variable to be optimized
            faces : List of indices of faces to be optimized
            
            
        """ 
        
        # Vertices indices
        v_idx = var_idx[var_name]
        
        # Filtrate the faces that have 4 vertices
        faces = [f for f in faces if len(f) == 4]

        # Init indices
        self.v0_idx  = []
        self.v1_idx  = []
        self.v2_idx  = []
        self.v3_idx  = []

        # Column indices
        num_faces = len(faces)

        self._rows_const = []
        self._cols_const = []
        self._values_const = []

        for i, f in enumerate(faces):

            # Get the indices of the vertices
            v0 = v_idx[interleave_indices_dim(f[0],3)]
            v1 = v_idx[interleave_indices_dim(f[1],3)]
            v2 = v_idx[interleave_indices_dim(f[2],3)]
            v3 = v_idx[interleave_indices_dim(f[3],3)]

            self.v0_idx.append(v0)
            self.v1_idx.append(v1)
            self.v2_idx.append(v2)
            self.v3_idx.append(v3)

            self._rows_const.extend( np.arange(3*i, 3*i + 3).repeat(4).tolist() )
                    
            for i in range(3):
                self._cols_const.extend([v0[i], v1[i], v2[i], v3[i]])
                self._values_const.extend([1.0, -1.0, 1.0, -1.0])
            

        self.num_residuals = num_faces * 3



    def grad(self, X):
    
        return None

    def res(self, X):
        v0 = X[self.v0_idx].reshape(-1, 3)
        v1 = X[self.v1_idx].reshape(-1, 3)
        v2 = X[self.v2_idx].reshape(-1, 3)
        v3 = X[self.v3_idx].reshape(-1, 3)

        res_par = v2 + v0 - v3 - v1

        
        return res_par.flatten()



class StraightAngle(ObjectiveTerm):

    def __init__(self) -> None:
        """ Rectangle: Energy term that minimize the deviation of a straigh angle given three points.
        The energy function is given by:
        Energy :   sum_triplet ( v1 - v0)/||v1-v0||.(v2 - v0)/||v2-v0|
          v1------------v0
                         |   
                         |   
                         |   
                         |
                        v2   
        """
        super().__init__()
        self.name = "Straight Angle" # Name of the constraint
        
        
      
    def initialize_objective(self, X, var_idx, var_name, v0_idx, v1_idx, v2_idx ) -> None:
        """ 
        Input:
            X : Variables
            var_idx : dictionary of indices of variables
            var_name: Name of the variable to be optimized
            v0_idx, v1_idx, v2_idx : List of indices of the three points to be optimized
        """ 

        # We must assert that v0 v1 v3 have the same length
        assert len(v0_idx) == len(v1_idx) == len(v2_idx), "The three list of points must have the same length"
        
        # Vertices indices
        v_idx = var_idx[var_name].reshape(-1, 3)
    
        # Init indices
        self.v0_idx  = v_idx[v0_idx]
        self.v1_idx  = v_idx[v1_idx]
        self.v2_idx  = v_idx[v2_idx]
        

        num_triplets = len(v0_idx)

        # We will have 9 residuals for each triplet (3 for each point) 
        self._rows = np.arange(num_triplets).repeat(9).tolist()
        self._cols = np.column_stack((self.v0_idx, self.v1_idx, self.v2_idx)).flatten().tolist()

        self.num_residuals = num_triplets 

        v0 = X[self.v0_idx]
        v1 = X[self.v1_idx]
        v2 = X[self.v2_idx]

        self.l10 = np.linalg.norm( v1 - v0, axis=1 )
        self.l20 = np.linalg.norm( v2 - v0, axis=1 )

    def grad(self, X):

        v0 = X[self.v0_idx]
        v1 = X[self.v1_idx]
        v2 = X[self.v2_idx]

        # E = (v2 - v0)/||v2-v0|| . (v1 - v0)/||v1-v0||

        denom = self.l10 * self.l20

        # dv1 
        dv1 = (v2 - v0) / denom[:, np.newaxis]

        dv2 = (v1 - v0) / denom[:, np.newaxis]

        dv0 = - dv1 - dv2

        vals = np.column_stack((dv0, dv1, dv2)).flatten()


        self.l10 = np.linalg.norm( v1 - v0, axis=1 )
        self.l20 = np.linalg.norm( v2 - v0, axis=1 )
    
        return vals

    def res(self, X):
        v0 = X[self.v0_idx]
        v1 = X[self.v1_idx]
        v2 = X[self.v2_idx]

        denom = self.l10 * self.l20

        res = np.sum((v2 - v0) * (v1 - v0), axis=1) / denom
        
        return res

class cyclideFairness(ObjectiveTerm):

    def __init__(self) -> None:
        """ Rectangle: Energy term that minimize the deviation of two edges directions.
        The energy function is given by:
        Energy :   sum_{two_edges} (a3 - a2 - a1 + a0)

        R_i  |  R_j
        a0  a1 a2 a3
        """
        super().__init__()
        self.name = "cyclideFairness" # Name of the constraint
        
        
      
    def initialize_objective(self, X, var_idx, var_name, a0_idx, a1_idx, a2_idx, a3_idx ) -> None:
        """ 
        Input:
            X : Variables
            var_idx : dictionary of indices of variables
            var_name: Name of the variable to be optimized
            a0_idx, a1_idx, a2_idx, a3_idx : List of indices of the four points to be optimized
        """ 

        # We must assert that a0 a1 a2 a3 have the same length
        assert len(a0_idx) == len(a1_idx) == len(a2_idx) == len(a3_idx), "The four list of points must have the same length"
        
        # Vertices indices
        v_idx = var_idx[var_name].reshape(-1, 3)
    
        # Init indices
        self.a0_idx  = v_idx[a0_idx]
        self.a1_idx  = v_idx[a1_idx]
        self.a2_idx  = v_idx[a2_idx]
        self.a3_idx  = v_idx[a3_idx]

        num_triplets = len(self.a0_idx)

        # E = (a3 - a2 - a1 + a0)
        # We will have 9 residuals for each triplet (3 for each point) 
        self._rows_const = np.arange(num_triplets*3).repeat(4).tolist()
        self._cols_const = []

        for i in range(num_triplets):
            self._cols_const.extend([self.a0_idx[i,0], self.a1_idx[i,0], self.a2_idx[i,0], self.a3_idx[i,0],
                                     self.a0_idx[i,1], self.a1_idx[i,1], self.a2_idx[i,1], self.a3_idx[i,1],
                                     self.a0_idx[i,2], self.a1_idx[i,2], self.a2_idx[i,2], self.a3_idx[i,2]])

        self._values_const = np.tile([1.0, -1.0, -1.0, 1.0], num_triplets*3).tolist()

        self.num_residuals = num_triplets * 3

        print("cols \t rows \t values")
        for i in range(5):
            print(f"{self._cols_const[i*4:(i+1)*4]} \t {self._rows_const[i*4:(i+1)*4]} \t {self._values_const[i*4:(i+1)*4]}")

        
        self.it = 0

    def grad(self, X):
        return None

    def res(self, X):
        a0 = X[self.a0_idx]
        a1 = X[self.a1_idx]
        a2 = X[self.a2_idx]
        a3 = X[self.a3_idx]

        # E = (a3 - a2 - a1 + a0)
        res = a3 - a2 - a1 + a0

        if self.it < 10 :
            self.w *= 0.9
        elif self.it >= 10 :
            self.w = 0 

        self.it += 1

        return res.flatten()

In [ ]:
# ── Cell 3 — Mesh + initial rectangle, with sliders ──────────────────────────
#
# The seed rectangle is centred on the corner face's barycentre, with u
# pointing toward the U-arm's first neighbour and v toward the V-arm's
# (seed_direction_towards). h_u/h_v are each bounded by the tightest edge on
# the strip they'll later be reflected across (docs/PROPAGATION_REDESIGN.md
# Step 1) so the seed cannot already outgrow a face it hasn't reached yet,
# then scaled by their own slider. theta rotates the frame on top of that;
# offset u/v move its centre within the face (clamped to stay inside).
MESH_NAME    = "cone_mesh_0.obj"   # cone_mesh_small · cone_mesh_0 · cone_mesh_small2
CORNER_INDEX = 0                   # which detected corner face to seed from

V, F = read_obj(os.path.join(DATA, MESH_NAME))
V = normalize_vertices(V)
F = np.array(F)

mesh = Mesh()
mesh.make_mesh(V, F)
f_normals = mesh.face_normals          # per-face; keep a pristine copy
faces     = mesh.faces
v_f_adj   = mesh.vertex_face_adjacency_list()
v_v_adj   = mesh.vertex_adjacency_list()
inn_v     = mesh.inner_vertices()

corner_faces = find_corner_faces(mesh)
SEED_FACE    = int(corner_faces[CORNER_INDEX])
grid_faces   = grid_coordinates(mesh.faces, seed_face=SEED_FACE)
R, C = grid_faces.shape

print(f"{MESH_NAME}:  |V|={len(V)}  |F|={len(F)}  interior vertices={len(inn_v)}")
print(f"corner faces: {corner_faces.tolist()}   seed: {SEED_FACE}   grid: {grid_faces.shape}")

ws.clear()
ws.register_surface_mesh("mesh", mesh.vertices, mesh.faces,
                         color=PALETTE["faces"], show_edges=True)

# u toward the U-arm's second face (grid_faces[1, 0]), v toward the V-arm's
# (grid_faces[0, 1]) — see docs/PROPAGATION_REDESIGN.md Step 1.
F1 = int(grid_faces[1, 0]) if R > 1 else -1
Fm = int(grid_faces[0, 1]) if C > 1 else -1
U_DIR = (seed_direction_towards(mesh, f_normals, SEED_FACE, F1) if F1 >= 0
         else unit(mesh.vertices[mesh.faces[SEED_FACE][1]]
                   - mesh.vertices[mesh.faces[SEED_FACE][0]]))
V_DIR = (seed_direction_towards(mesh, f_normals, SEED_FACE, Fm) if Fm >= 0
         else np.cross(unit(f_normals[SEED_FACE]), U_DIR))

# Bound h_u/h_v so the seed cannot outgrow the tightest face it will later be
# reflected across: h_u from the U-arm faces' edges aligned with v, h_v from
# the V-arm faces' edges aligned with u.
strip_a, strip_b = mesh.quad_face_strips(SEED_FACE)
if F1 >= 0 and F1 in strip_a:
    U_STRIP, V_STRIP = strip_a, strip_b
elif F1 >= 0 and F1 in strip_b:
    U_STRIP, V_STRIP = strip_b, strip_a
else:
    U_STRIP, V_STRIP = strip_a, strip_b
H_U_MAX = strip_min_edge_length(mesh, U_STRIP, V_DIR, f_normals)
H_V_MAX = strip_min_edge_length(mesh, V_STRIP, U_DIR, f_normals)
print(f"h_u_max={H_U_MAX:.6f}  h_v_max={H_V_MAX:.6f}")

seed_frames = {}       # (face_id, vertex_id) -> {"pt", "u", "v", "n", "f"}
seed_out = widgets.Output()


def rebuild_seed_rectangle(theta, h_u_factor, h_v_factor, offset_u, offset_v):
    """Rebuild the seed rectangle; Cell 4 propagates from whatever is set here."""
    global seed_frames

    seed_frames = seed_rectangle_at_barycentre(
        mesh, f_normals, SEED_FACE, U_DIR,
        h_u_factor * H_U_MAX, h_v_factor * H_V_MAX,
        theta=theta, offset_u=offset_u, offset_v=offset_v)

    pts, w, h = get_rectangle(mesh, seed_frames, SEED_FACE)

    ws.register_point_cloud("Seed point", np.array([pts.mean(axis=0)]),
                            color=PALETTE["seed"], radius=0.001)
    ws.register_surface_mesh("Seed rectangle", pts, np.array([[0, 1, 2, 3]]),
                             color=PALETTE["rectangle"])

    seed_out.clear_output(wait=True)
    with seed_out:
        print(f"theta={theta:.3f}   seed rectangle: w={w:.6f}  h={h:.6f}")


sliders = dict(
    theta      = widgets.FloatSlider(value=0.0, min=0.0,  max=2 * np.pi, step=0.01,
                                     description="theta",      continuous_update=True),
    h_u_factor = widgets.FloatSlider(value=0.5, min=0.05, max=1.0,       step=0.01,
                                     description="h_u factor", continuous_update=True),
    h_v_factor = widgets.FloatSlider(value=0.5, min=0.05, max=1.0,       step=0.01,
                                     description="h_v factor", continuous_update=True),
    offset_u   = widgets.FloatSlider(value=0.0, min=-0.5, max=0.5,       step=0.01,
                                     description="offset u",   continuous_update=True),
    offset_v   = widgets.FloatSlider(value=0.0, min=-0.5, max=0.5,       step=0.01,
                                     description="offset v",   continuous_update=True),
)
for s in sliders.values():
    s.layout = widgets.Layout(width="480px")

# interactive_output fires once immediately (building the initial rectangle)
# and again on every slider drag.
_link = widgets.interactive_output(rebuild_seed_rectangle, sliders)
display(*sliders.values(), seed_out)

In [ ]:
# ── Cell 4 — Full pipeline: propagate → patches → cones → optimize → repeat ──
#
# Everything from the seed rectangle of Cell 3 through to the optimized
# geometry. Each stage's merged meshes are stored in `data` so Cell 5 can
# export them without recomputing.
#
#   data["init_*"]  geometry from the propagated (pre-optimization) corners
#   data["opt_*"]   the same, rebuilt from the optimized corners
#
# See docs/PROPAGATION_REDESIGN.md for the algorithm and the energy terms.
from hanan.optimization.optimizer import Optimizer
from hanan.optimization.cyclicity import Cyclicity
from hanan.optimization.planarity import Planarity

U_DIST_FACTOR = 0.5
V_DIST_FACTOR = 0.5
PATCH_N = PATCH_M = 10
N_OPT_STEPS   = 30

data = {}


# ── helpers ──────────────────────────────────────────────────────────────────

def fit_circle_4pts(pts):
    """Least-squares best-fit circle through 4 (possibly non-coplanar) points.

    Seeds the plane from circle_3pts on the first three, then refines by SVD
    over all four so the residual is spread evenly instead of forcing three
    points to be exact.
    """
    pts = np.asarray(pts, dtype=float)
    c0, n0, r0 = circle_3pts(pts[0], pts[1], pts[2])
    normal0 = n0[0]

    centroid = pts.mean(axis=0)
    A = pts - centroid
    _, _, Vt = np.linalg.svd(A)
    normal = Vt[-1]
    if normal @ normal0 < 0:
        normal = -normal

    ref = np.array([1.0, 0.0, 0.0]) if abs(normal[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    e1 = unit(ref - (ref @ normal) * normal)
    e2 = np.cross(normal, e1)

    x, y = A @ e1, A @ e2
    M = np.stack([x, y, np.ones_like(x)], axis=1)
    b = -(x ** 2 + y ** 2)
    D, E, Fc = np.linalg.lstsq(M, b, rcond=None)[0]
    cx, cy = -D / 2, -E / 2
    radius = np.sqrt(cx ** 2 + cy ** 2 - Fc)

    center = centroid + cx * e1 + cy * e2
    d_plane = A @ normal
    radial = np.sqrt((x - cx) ** 2 + (y - cy) ** 2) - radius
    return center, normal, radius, d_plane, radial


def _arc_bezier(p0, p2, n0, n2, eps=1e-12):
    """Middle control point and weight of the circular arc p0 -> p2.

    A boundary of a principal cyclidic patch is a circular arc, which has an
    exact rational quadratic Bezier form: control points (p0, b1, p2) with
    weights (1, cos(theta/2), 1) for central angle theta.

    b1 is built from the arc's OWN CIRCLE rather than by intersecting the two
    endpoint tangent lines. On a near-conical mesh adjacent corner normals are
    nearly equal, so those tangents are nearly parallel (measured: |t0.t2| >
    0.99 at every one of the 441 interior vertices of cone_mesh_0, and
    1.0 - 1e-8 at several) and the intersection is ill-conditioned — the
    least-squares solution ran away to 882x the chord length at vertex 36,
    and ~1.5-1.9x at vertices 202/203, producing the far-off points in the
    rendered patches.

    The circle is well defined regardless: the centre lies along the in-plane
    direction perpendicular to the tangent at p0, at the distance that makes
    it equidistant from p0 and p2,

        t0    = normalize( n0 x (d x n0) )          tangent at p0, d = chord dir
        axis  = normalize( t0 x chord )             arc-plane normal
        rdir  = normalize( axis x t0 )              in-plane, perp. to tangent
        rho   = |chord|^2 / (2 chord.rdir)          signed centre distance
        c     = p0 + rho rdir,   R = |rho|
        w1    = cos(theta/2),    sin(theta/2) = |chord| / 2R
        b1    = c + (R / w1) * normalize(midpoint - c)

    A genuinely straight edge (tangent parallel to the chord, or R -> inf)
    falls back to the chord midpoint with weight 1, which is exactly the
    degree-2 form of a segment.
    """
    p0 = np.asarray(p0, float)
    p2 = np.asarray(p2, float)
    n0 = unit(np.asarray(n0, float))

    chord = p2 - p0
    L = np.linalg.norm(chord)
    mid = (p0 + p2) / 2.0
    if L < eps:
        return mid, 1.0
    d = chord / L

    t0 = np.cross(n0, np.cross(d, n0))
    nt = np.linalg.norm(t0)
    if nt < 1e-12:                       # normal parallel to the chord
        return mid, 1.0
    t0 = t0 / nt
    if t0 @ d < 0:
        t0 = -t0

    axis = np.cross(t0, chord)
    na = np.linalg.norm(axis)
    if na < 1e-14:                       # tangent parallel to chord: straight
        return mid, 1.0
    rdir = np.cross(axis / na, t0)
    rdir = rdir / np.linalg.norm(rdir)

    denom = 2.0 * (chord @ rdir)
    if abs(denom) < eps * max(1.0, L):
        return mid, 1.0
    rho = (L * L) / denom
    c = p0 + rho * rdir
    R = abs(rho)

    w1 = float(np.cos(np.arcsin(np.clip(L / (2.0 * R), -1.0, 1.0))))
    if w1 < eps:
        return mid, 1.0

    mhat = mid - c
    nm = np.linalg.norm(mhat)
    if nm < eps:                         # half circle: centre at the midpoint
        return mid, 1.0
    return c + (R / w1) * (mhat / nm), w1


def dupin_patch_bezier(corners, normals, N=10, M=10):
    """Degree (2,2) rational Bezier cyclidic patch (Pottmann et al., Prop. 8).

    The four corners ARE the control points b00, b20, b22, b02, so the patch
    interpolates them exactly — unlike the Lie-sphere construction, where x12
    is recovered as the meeting point of two cyclidic families and absorbs the
    accumulated propagation error.
    """
    p00, p20, p22, p02 = [np.asarray(p, float) for p in corners]
    n00, n20, n22, n02 = [unit(np.asarray(n, float)) for n in normals]

    b10, w10 = _arc_bezier(p00, p20, n00, n20)
    b21, w21 = _arc_bezier(p20, p22, n20, n22)
    b12, w12 = _arc_bezier(p02, p22, n02, n22)
    b01, w01 = _arc_bezier(p00, p02, n00, n02)

    # b11 = intersection of the tangent planes at the four corners
    A = np.array([n00, n20, n22, n02])
    rhs = np.array([n00 @ p00, n20 @ p20, n22 @ p22, n02 @ p02])
    b11, *_ = np.linalg.lstsq(A, rhs, rcond=None)

    B = np.array([[p00, b01, p02],
                  [b10, b11, b12],
                  [p20, b21, p22]], dtype=float)
    W = np.array([[1.0, w01, 1.0],
                  [w10, w10 * w01, w12],
                  [1.0, w21, 1.0]], dtype=float)

    def bern2(t):
        return np.array([(1 - t) ** 2, 2 * t * (1 - t), t ** 2])

    verts = []
    for s in np.linspace(0, 1, N):
        Bs = bern2(s)
        for t in np.linspace(0, 1, M):
            coef = np.outer(Bs, bern2(t)) * W
            verts.append(np.einsum("ij,ijk->k", coef, B) / coef.sum())

    quads = [[i * M + j, i * M + j + 1, (i + 1) * M + j + 1, (i + 1) * M + j]
             for i in range(N - 1) for j in range(M - 1)]
    return np.array(verts), np.array(quads)


def bezier_boundary_curves(BV, N, M):
    """The four boundary curves of a dupin_patch_bezier grid.

    Keyed the same way dupin_patch_boundary_curves keys the Bobenko grid — by
    ring position (0=x0, 1=x1, 2=x12, 3=x2) — but the two grids are sampled
    differently, so the slices differ. dupin_patch_bezier iterates s outer /
    t inner, giving (verified) BV[0]=x0, BV[M-1]=x2, BV[(N-1)*M]=x1,
    BV[N*M-1]=x12. Hence:

        (0,1) x0 -> x1  : column j=0      -> BV[0::M]
        (0,3) x0 -> x2  : row    i=0      -> BV[0:M]
        (1,2) x1 -> x12 : row    i=N-1    -> BV[(N-1)*M:]
        (2,3) x12 -> x2 : column j=M-1    -> BV[M-1::M]
    """
    BV = np.asarray(BV)
    return {
        (0, 1): BV[0::M],
        (0, 3): BV[0:M],
        (1, 2): BV[(N - 1) * M:],
        (2, 3): BV[M - 1::M],
    }


def build_all(frames, tag):
    """Rectangles, Dupin patches (both methods), cone strips, circumcircles.

    `frames` is {(face_id, vertex_id): frame}. Everything is merged and stored
    under data["<tag>_*"].
    """
    # -- rectangles --
    Vl, Fl = [], []
    placed = []
    for f_id in range(len(faces)):
        try:
            pts, _, _ = get_rectangle(mesh, frames, f_id)
        except DupinPropagationError:
            continue
        Vl.append(np.asarray(pts))
        Fl.append(np.array([[0, 1, 2, 3]]))
        placed.append(f_id)
    data[f"{tag}_rectangles"] = merge_meshes(Vl, Fl) if Vl else None

    # -- Dupin patches: Bobenko/Lie-sphere and Bezier, plus circumcircles --
    Vb, Fb = [], []          # Bobenko
    Vz, Fz = [], []          # Bezier
    circles = []
    dupin_curves = {}        # patch boundaries, Bobenko grid
    bez_curves = {}          # patch boundaries, Bezier grid
    dupin_vertices = []
    lie_gaps = []

    for vtx in inn_v:
        vtx = int(vtx)
        adj = [int(f) for f in v_f_adj[vtx]]
        if len(adj) != 4:
            continue
        fids = [(f, vtx) for f in adj]   # frames are keyed (face_id, vertex_id)
        if any(fid not in frames for fid in fids):
            continue

        Fs  = [frames[fid] for fid in fids]
        pts = [np.asarray(f["pt"], float) for f in Fs]
        nrm = [unit(np.asarray(f["n"], float)) for f in Fs]

        # circumcircle through the four cell corners
        center, normal, radius, _, _ = fit_circle_4pts(pts)
        circles.append(circle_3d(center, normal, radius, pts=48))

        # (a) Bobenko / Lie-sphere
        F0, F1_, F12, F2 = Fs
        try:
            DV, DF = Dupin_patch(F0, F1_, F12, F2, N=PATCH_N, M=PATCH_M)
            DV = np.asarray(DV)
            Vb.append(DV)
            Fb.append(np.asarray(DF))
            lie_gaps.append(np.linalg.norm(DV[PATCH_N * PATCH_M - 1]
                                           - np.asarray(F12["pt"], float)))
            curves = dupin_patch_boundary_curves(DV, PATCH_N, PATCH_M)
            dupin_curves[vtx] = {frozenset((adj[a], adj[b])): c
                                 for (a, b), c in curves.items()}
            dupin_vertices.append(vtx)
        except Exception as e:
            print(f"  [{tag}] vertex {vtx}: Bobenko patch failed — "
                  f"{type(e).__name__}: {e}")

        # (b) Bezier
        BV, BF = dupin_patch_bezier(pts, nrm, N=PATCH_N, M=PATCH_M)
        Vz.append(BV)
        Fz.append(BF)
        bez_curves[vtx] = {frozenset((adj[a], adj[b])): c
                           for (a, b), c in
                           bezier_boundary_curves(BV, PATCH_N, PATCH_M).items()}

    edge_v1, edge_v2 = mesh.edge_vertices()

    data[f"{tag}_cyclides"]        = merge_meshes(Vb, Fb) if Vb else None
    data[f"{tag}_cyclides_bezier"] = merge_meshes(Vz, Fz) if Vz else None
    data[f"{tag}_circles"]         = merge_curves(circles) if circles else None

    # -- cone strips joining neighbouring patch boundaries --
    # Built separately per patch method: a cone strip is only meaningful when
    # both sides come from the SAME construction, otherwise the two arcs
    # belong to different surfaces and the strip bridges the gap between them.
    n_cones = {}
    for method, curve_map in (("cones", dupin_curves), ("cones_bezier", bez_curves)):
        Vs_, Fs_ = [], []
        for vi, vj in zip(edge_v1.tolist(), edge_v2.tolist()):
            fa, fb = mesh.faces_from_edge(vi, vj)
            if fa < 0 or fb < 0:            # boundary edge — only one face
                continue
            key = frozenset((int(fa), int(fb)))
            ci, cj = curve_map.get(vi), curve_map.get(vj)
            if ci is None or cj is None or key not in ci or key not in cj:
                continue
            arc_i = np.asarray(ci[key])
            arc_j = np.asarray(cj[key])
            # the two patches order their rings independently, so arc_j may
            # run the other way — pick the orientation that matches the ends
            same    = (np.linalg.norm(arc_i[0] - arc_j[0])
                       + np.linalg.norm(arc_i[-1] - arc_j[-1]))
            flipped = (np.linalg.norm(arc_i[0] - arc_j[-1])
                       + np.linalg.norm(arc_i[-1] - arc_j[0]))
            if flipped < same:
                arc_j = arc_j[::-1]
            Vq, Fq = ruled_strip(arc_i, arc_j)
            Vs_.append(np.asarray(Vq))
            Fs_.append(np.asarray(Fq))
        data[f"{tag}_{method}"] = merge_meshes(Vs_, Fs_) if Vs_ else None
        n_cones[method] = len(Vs_)

    lie_gaps = np.array(lie_gaps) if lie_gaps else np.zeros(1)
    print(f"[{tag}] rectangles={len(placed)}  cyclides={len(Vb)}  "
          f"bezier={len(Vz)}  cones={n_cones['cones']}  "
          f"cones_bezier={n_cones['cones_bezier']}  circles={len(circles)}")
    print(f"[{tag}] Bobenko x12 gap: mean={lie_gaps.mean():.3e}  max={lie_gaps.max():.3e}")
    return placed


# ── 1. propagate from the seed ───────────────────────────────────────────────
frames, rectangles_to_draw, prop_stats = propagate_rectangles(
    mesh, f_normals, grid_faces, seed_frames,
    u_factor=U_DIST_FACTOR, v_factor=V_DIST_FACTOR)

for arm, fa, fb, msg in prop_stats["arm_stopped"]:
    print(f"{arm} arm stopped at {fa}->{fb}: {msg}")
if prop_stats["interior_skipped"]:
    print(f"interior cells skipped: {len(prop_stats['interior_skipped'])}")

print(f"propagated rectangles: {len(rectangles_to_draw)} / {len(F)}")

build_all(frames, "init")

# ── 2. index sets for the optimizer ──────────────────────────────────────────
rectangle_corners = []
dict_indices = {}
idx = 0
for f_id, vertices_id in enumerate(faces):
    for v in vertices_id:
        dict_indices[(f_id, v)] = idx
        idx += 1
        rectangle_corners.append(frames[(f_id, v)]["pt"])
rectangle_corners = np.array(rectangle_corners)

rectangle_faces = np.array([[dict_indices[(f_id, v)] for v in face]
                            for f_id, face in enumerate(faces)])
circles_faces = np.array([[dict_indices[(f_id, vtx)] for f_id in v_f_adj[vtx]
                           if (f_id, vtx) in dict_indices] for vtx in inn_v])

#v_f_adj   = mesh.vertex_face_adjacency_list()
#v_v_adj   = mesh.vertex_vertex_adjacency_list()
#inn_v     = mesh.inner_vertices()

#Fairness index getting
a0 = []
a1 = []
a2 = []
a3 = []

for v_id in inn_v:

    # Get the adjacent faces of the vertex
    incident_faces = v_f_adj[v_id]
    neighbor_vertices = v_v_adj[v_id]

    # Check if the vertex has exactly 4 incident faces
    if len(incident_faces) == 4:
        for i in range(4):
            fi = incident_faces[i]
            vi = neighbor_vertices[i]
            fi1 = incident_faces[(i + 1) % 4]
            vi2 = neighbor_vertices[(i + 2) % 4]

            a0.append(dict_indices[(fi, vi)])
            a1.append(dict_indices[(fi, v_id)])
            a2.append(dict_indices[(fi1, v_id)])
            a3.append(dict_indices[(fi1, vi2)])


print(f"rectangle_faces {rectangle_faces.shape}   circles_faces {circles_faces.shape}")

# ── 3. optimize ──────────────────────────────────────────────────────────────
opt = Optimizer()
weights = {"planarity": 1.0, "straight_angle": 1.0,
           "cyclicity": 1.0, "conicality": 1.0, "fairness": 0.00}

opt.add_variable("p",   rectangle_corners.flatten())
opt.add_variable("r_n", f_normals.flatten())

# auxiliary cone axis per interior vertex
aux_n = []

for v in inn_v:
    incident_faces = v_f_adj[v]
    if len(incident_faces) == 4:
       n0, n1, n2, n3 = [f_normals[f] for f in incident_faces] 
       normal_n = unit(np.cross(n2 - n0, n3 - n1))
       aux_n.append(normal_n)
    else:
        print(f"Vertex {v} does not have 4 incident faces, skipping aux_n computation.")


opt.add_variable("aux_n", np.array(aux_n).flatten())

# auxiliary circumcentre/normal per interior vertex
cy_c, cy_n = [], []
for cf in circles_faces:
    c, n, r, _, _ = fit_circle_4pts(np.array([rectangle_corners[i] for i in cf]))
    cy_c.append(c)
    cy_n.append(n)
opt.add_variable("cy_c", np.array(cy_c).flatten())
opt.add_variable("cy_n", np.array(cy_n).flatten())

# StraightAngle needs (v0, v1, v3) triples — the two edges meeting at v0
sa_v0 = [f[0] for f in rectangle_faces]
sa_v1 = [f[1] for f in rectangle_faces]
sa_v2 = [f[3] for f in rectangle_faces]

sa_v0.extend([f[2] for f in rectangle_faces])
sa_v1.extend([f[1] for f in rectangle_faces])
sa_v2.extend([f[3] for f in rectangle_faces])

cyclicity = Cyclicity();       cyclicity.name = "cyclicity"
opt.add_objective_term(cyclicity, args=(circles_faces, "p", "cy_c", "cy_n"),
                       w=weights["cyclicity"], ce=1)

plan = Planarity(); plan.name = "planarity"
opt.add_objective_term(plan, args=(rectangle_faces, "p", "r_n"),
                       w=weights["planarity"], ce=1)

straight_angle = StraightAngle(); straight_angle.name = "straight_angle"
opt.add_objective_term(straight_angle, args=("p", sa_v0, sa_v1, sa_v2),
                       w=weights["straight_angle"], ce=1)

# conicality: consecutive face normals around a vertex lie on a cone about aux_n
conicality = Planarity();      conicality.name = "conicality"
opt.add_objective_term(conicality, args=([v_f_adj[v] for v in inn_v], "r_n", "aux_n"),
                       w=weights["conicality"], ce=1)

# Fairness: minimize the deviation of two edges directions
fairness = cyclideFairness();  fairness.name = "fairness"
opt.add_objective_term(fairness, args=("p", a0, a1, a2, a3), w=weights["fairness"], ce=1)

opt.unitize_variable("cy_n", 3, 10)
opt.unitize_variable("aux_n", 3, 10)
opt.unitize_variable("r_n", 3, 10)
opt.initialize_optimizer(verbose=True, adaptive_mu=False)

for _ in range(N_OPT_STEPS):
    opt.get_gradients()
    opt.optimize_step()
opt.print_report()

new_rectangle_corners = opt.unpack("p").reshape(-1, 3)

# ── 4. rebuild everything from the optimized corners ─────────────────────────
# The propagated `frames` are stale now: the optimizer moved the corners.
opt_frames = build_rectangle_frames(new_rectangle_corners, rectangle_faces,
                                    mesh.face_normals, mesh_faces=faces)

build_all(opt_frames, "opt")

# ── 5. push both passes to the viewer ────────────────────────────────────────
# init and opt are registered as separate objects with distinct colours, so
# they can be toggled independently in the scene panel to compare.
ws.remove("Seed rectangle")
# "cones":           "Cones (Bobenko)",
# "cyclides":        "Cyclides (Bobenko)",
LABELS = {
    "rectangles":      "Rectangles",
    "cyclides_bezier": "Cyclides (Bezier)",
    "cones_bezier":    "Cones (Bezier)",
}

for tag in ("init", "opt"):
#for tag in ("opt"):
    for kind, label in LABELS.items():
        key = f"{tag}_{kind}"
        if data.get(key) is None:
            continue
        Vm, Fm = data[key]
        if tag=="opt" and kind not in ["Cones (Bobenko)", "Cyclides (Bobenko)"]:
            ws.register_surface_mesh(f"{label} [{tag}]", Vm, Fm,
                                 color=PASS_COLORS[tag][kind])

    key = f"{tag}_circles"
    if data.get(key) is not None:
        Vc, Ec = data[key]
        ws.register_curve_network(f"Circumcircles [{tag}]", Vc, Ec,
                                  color=PASS_COLORS[tag]["circles"],
                                  radius=0.0005)

ws.show()

print("\nstored in `data`:", ", ".join(sorted(data)))

In [ ]:
# ── Cell 5 — Export ──────────────────────────────────────────────────────────
# Writes everything Cell 4 stored in `data` to OUT_DIR, as
#
#   init_rectangles.obj        opt_rectangles.obj
#   init_cyclides.obj          opt_cyclides.obj           (Bobenko / Lie-sphere)
#   init_cyclides_bezier.obj   opt_cyclides_bezier.obj    (rational Bezier)
#   init_cones.obj             opt_cones.obj              (Bobenko boundaries)
#   init_cones_bezier.obj      opt_cones_bezier.obj       (Bezier boundaries)
#   init_circles.obj           opt_circles.obj            (circumcircles, polylines)
#
# plus the base mesh. Nothing is recomputed here.


def write_curve_obj(filename, vertices, edges):
    """Write a polyline / curve network as OBJ 'l' (line) elements."""
    with open(str(filename), "w") as fh:
        for v in vertices:
            fh.write("v {} {} {}\n".format(v[0], v[1], v[2]))
        for e in edges:
            fh.write("l {} {}\n".format(int(e[0]) + 1, int(e[1]) + 1))


written = []

for tag in ("init", "opt"):
    for kind in ("rectangles", "cyclides", "cyclides_bezier",
                 "cones", "cones_bezier"):
        key = f"{tag}_{kind}"
        if data.get(key) is None:
            continue
        Vm, Fm = data[key]
        path = os.path.join(OUT_DIR, f"{key}.obj")
        write_obj(path, Vm, Fm)
        written.append((f"{key}.obj", len(Vm), len(Fm), "faces"))

    key = f"{tag}_circles"
    if data.get(key) is not None:
        Vc, Ec = data[key]
        path = os.path.join(OUT_DIR, f"{key}.obj")
        write_curve_obj(path, Vc, Ec)
        written.append((f"{key}.obj", len(Vc), len(Ec), "edges"))

write_obj(os.path.join(OUT_DIR, "cone_mesh.obj"), mesh.vertices, mesh.faces)
written.append(("cone_mesh.obj", len(mesh.vertices), len(mesh.faces), "faces"))

for name, nv, nf, what in written:
    print(f"  {name:<28} {nv:>7} verts  {nf:>7} {what}")
print(f"\nwritten to {OUT_DIR}")